# Week 10: MLOps, System Evaluation & Explainable AI
## Notebook 08: Production Data Drift Metrics: PSI & Wasserstein Distance

### 1. Conceptual Intuition & Theoretical Background
Machine learning models degrade silently in production when the serving data distribution $Q(X)$ drifts away from the training distribution $P(X)$ (**Covariate Shift**).

Key mathematical drift metrics:
1. **Population Stability Index (PSI):**
Measures relative divergence across $B$ quantile bins:
$$\text{PSI} = \sum_{b=1}^{B} \left( Q_b - P_b \right) \times \ln\left( \frac{Q_b}{P_b} \right)$$
- $\text{PSI} < 0.10$: No significant change; model is safe.
- $0.10 \le \text{PSI} \le 0.25$: Moderate drift; warning flag.
- $\text{PSI} > 0.25$: Significant drift; retrain model immediately.

2. **Wasserstein-1 Distance (Earth Mover's Distance):**
Measures minimum work required to transform distribution $P$ into $Q$:
$$W_1(P, Q) = \int_{-\infty}^{\infty} |F_P(x) - F_Q(x)| dx$$
Unlike KL divergence, Wasserstein distance remains well-defined even when distribution supports do not overlap.

---

### 2. Scratch Python Implementation: Production Drift Suite

In [ ]:
import numpy as np

def calculate_psi(baseline, target, num_bins=10):
    """
    Computes Population Stability Index (PSI) between baseline and production.
    """
    # Create quantile bins using baseline distribution
    percentiles = np.linspace(0, 100, num_bins + 1)
    bin_edges = np.percentile(baseline, percentiles)
    bin_edges[0] = -np.inf
    bin_edges[-1] = np.inf
    
    # Calculate empirical frequencies
    baseline_counts, _ = np.histogram(baseline, bins=bin_edges)
    target_counts, _ = np.histogram(target, bins=bin_edges)
    
    # Convert to fractions with epsilon smoothing to prevent log(0)
    P = (baseline_counts / len(baseline)) + 1e-6
    Q = (target_counts / len(target)) + 1e-6
    
    # Normalize fractions
    P /= np.sum(P)
    Q /= np.sum(Q)
    
    # PSI formula
    psi_value = np.sum((Q - P) * np.log(Q / P))
    return psi_value

def calculate_wasserstein_1d(u_samples, v_samples):
    """
    Computes 1D Wasserstein distance between two empirical distributions.
    """
    u_sorted = np.sort(u_samples)
    v_sorted = np.sort(v_samples)
    
    # Evaluate over unified grid
    all_values = np.sort(np.concatenate([u_sorted, v_sorted]))
    
    # Empirical CDFs
    u_cdf = np.searchsorted(u_sorted, all_values, side='right') / len(u_sorted)
    v_cdf = np.searchsorted(v_sorted, all_values, side='right') / len(v_sorted)
    
    # Integrated absolute difference
    deltas = np.diff(all_values)
    w1 = np.sum(np.abs(u_cdf[:-1] - v_cdf[:-1]) * deltas)
    return w1

# Synthetic Benchmark
np.random.seed(42)
baseline_data = np.random.normal(loc=50.0, scale=10.0, size=5000)

# Scenario 1: Stationary Production Traffic
prod_stationary = np.random.normal(loc=50.1, scale=9.9, size=2000)
# Scenario 2: Severe Covariate Shift
prod_drifted = np.random.normal(loc=58.0, scale=14.0, size=2000)

psi_clean = calculate_psi(baseline_data, prod_stationary)
psi_drift = calculate_psi(baseline_data, prod_drifted)

w1_clean = calculate_wasserstein_1d(baseline_data, prod_stationary)
w1_drift = calculate_wasserstein_1d(baseline_data, prod_drifted)

print(f"Stationary Traffic -> PSI: {psi_clean:.4f} (Status: Normal) | W1: {w1_clean:.3f}")
print(f"Drifted Traffic    -> PSI: {psi_drift:.4f} (Status: SEVERE DRIFT > 0.25) | W1: {w1_drift:.3f}")

### 3. Step-by-Step Code Walkthrough
1. **Quantile Binning**: Bins are fixed using baseline distributions to evaluate real production shifts.
2. **Epsilon Smoothing**: Adds $10^{-6}$ to prevent infinite values when target bins are empty.
3. **Threshold Alerting**: $\text{PSI} = 0.65$ triggers automated training pipeline retraining.

---

### 4. Industry Applications & Trade-offs
- **Credit Scoring & Fraud Engines:** Automated monitors (e.g. Evidently AI / Arize) flag changes in applicant income distributions.
- **Trade-off:** Binned metrics (PSI) can be sensitive to the number of chosen bins ($B$).

---

### 5. Top Interview Questions & Answers
**Q1: What is the difference between Data Drift (Covariate Shift) and Concept Drift?**
> **Answer:** Covariate Shift occurs when the feature distribution $P(X)$ changes while the true mapping $P(Y|X)$ remains constant. Concept Drift occurs when the underlying ground truth relationship $P(Y|X)$ changes (e.g. consumer purchasing habits change after a macroeconomic shock).

**Q2: Why is Wasserstein Distance preferred over KL Divergence for continuous drift detection?**
> **Answer:** KL divergence approaches infinity when distributions have disjoint or non-overlapping supports. Wasserstein distance reflects the physical distance required to shift probability mass, remaining smooth and well-behaved.